# ABW Step-by-Step Debug Notebook

This notebook is for **finer control and debugging** and makes the boundary explicit:

- BayesFlow: amortizer setup + training
- ABW: PSIS/MCMC workflow orchestration and diagnostics

Instead of calling `workflow.run(...)` end-to-end, it runs each step manually.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys

os.environ["JAX_ENABLE_X64"] = "False"
os.environ["KERAS_BACKEND"] = "jax"
# sys.path.insert(0, "/Users/lichengk/project/sbi_mcmc/BayesFlow")
# sys.path.insert(
#     0,
#     "/Users/lichengk/project_results/amortized_bayesian_workflow/code-release/src",
# )

In [ ]:
import numpy as np
import bayesflow as bf
import matplotlib.pyplot as plt

from amortized_bayesian_workflow import (
    ArtifactLayout,
    InferenceConfig,
    InferenceRunner,
)
from amortized_bayesian_workflow.approximators import (
    BayesFlowAmortizedPosterior,
)
from amortized_bayesian_workflow.backends import run_mcmc
from amortized_bayesian_workflow.psis import (
    compute_psis,
    resample_with_weights,
)

print(f"BayesFlow version: {bf.__version__}")

## 1. Build task and BayesFlow amortizer


In [ ]:
from amortized_bayesian_workflow.tasks.examples import (
    PsychometricTask,
    GeneralizedExtremeValue,
)

task = PsychometricTask()
task.var_dims


In [ ]:
from amortized_bayesian_workflow import configure_logging

# One line controls ABW + PyMC forward-sampling verbosity for this session.
configure_logging("WARNING")

In [ ]:
from pathlib import Path

layout = ArtifactLayout(
    root=Path("./artifacts"),
    task_name=task.task_name,
)
layout.ensure()

In [ ]:
from bayesflow.workflows import BasicWorkflow
from bayesflow import OfflineDataset
import random
import keras
from amortized_bayesian_workflow.approximators import (
    BayesFlowAmortizedPosterior,
)

num_train_simulations = 10000
num_validation_simulations = 500
batch_size = 256
epochs = 3
seed = 2026

# Seed all relevant RNGs before creating networks / training
keras.utils.set_random_seed(seed)

adapter = (
    bf.adapters.Adapter()
    .to_array()
    .convert_dtype("float64", "float32")
    .rename("parameters", "inference_variables")
    .rename("observables", "summary_variables")
)
inference_network = bf.networks.CouplingFlow()
summary_network = bf.networks.DeepSet()
if task.task_name == "GeneralizedExtremeValue":
    print(task.task_name)
    adapter = adapter.expand_dims("summary_variables", axis=-1)
    summary_network = bf.networks.DeepSet(
        summary_dim=8,
        depth=1,
        activation="mish",
        inner_pooling="max",
        output_pooling="mean",
        dropout=0.05,
    )

basic_amortized_workflow = BasicWorkflow(
    adapter=adapter,
    inference_network=inference_network,
    summary_network=summary_network,
    checkpoint_filepath=layout.models_dir / "training_checkpoint.pt",
)

train_sims = task.simulate_prior_predictive(num_train_simulations, seed=seed)
val_sims = task.simulate_prior_predictive(
    num_validation_simulations, seed=seed + 1
)

history = basic_amortized_workflow.fit_offline(
    data=train_sims,
    validation_data=val_sims,
    epochs=epochs,
    batch_size=batch_size,
)


In [ ]:
# Wrap the trained approximator and cache training summaries for Mahalanobis calibration.
approximator = BayesFlowAmortizedPosterior(
    basic_amortized_workflow.approximator
)
approximator.store_training_summaries(train_sims["observables"])
approximator.training_summaries.shape

In [ ]:
# obs = np.asarray(train_sims["observables"])
# if obs.ndim == 1:
#     obs = obs[None, ...]

# resolved_conditions, adapted, summary_outputs = (
#     approximator.approximator._prepare_conditions(
#         {approximator.observable_key: obs}
#     )
# )

In [ ]:
test_sims = task.simulate_prior_predictive(200)
basic_amortized_workflow.plot_default_diagnostics(test_sims)

In [ ]:
approximator.save(layout.models_dir)
approximator.load(layout.models_dir)

## 2. Build the InferenceRunner

With the amortizer trained and training summaries cached, construct the `InferenceRunner`.
The runner will use the pre-computed `training_summaries` directly for Mahalanobis
calibration — no re-simulation needed.


In [ ]:
config = InferenceConfig(seed=2026)
runner = InferenceRunner(task=task, approximator=approximator, config=config)
# Mahalanobis reference is auto-calibrated from approximator.training_summaries
# on the first call to runner.run() / _ensure_mahalanobis_reference().
# You can also trigger it explicitly:
runner._calibrate_mahalanobis_reference()
runner._mahalanobis_reference

## 3. Amortized posterior draws (manual)


In [ ]:
from amortized_bayesian_workflow.utils import read_from_file

test_observations = read_from_file(task.task_info_dir / "test_dataset.pkl")[
    :10
]

In [ ]:
# test_observations = task.simulate_prior_predictive(10, seed=config.seed + 42)[
#     "observables"
# ]
observation = test_observations[0]
amortized = approximator.sample_and_log_prob(
    observation, num_samples=config.num_amortized_draws, seed=config.seed
)
q_samples = amortized.samples
log_q = amortized.log_prob
q_samples.shape, log_q.shape

In [ ]:
from corner import corner

_ = corner(q_samples, show_titles=True)

## 4. Mahalanobis-distance diagnostic (observation-level)

A simple practical OOD-style diagnostic is to compare the observed dataset to prior-predictive simulated observations using Mahalanobis distance on flattened observables.

This is not the only possible diagnostic, but it is often useful before PSIS/MCMC.


In [ ]:
all_summaries = runner._summary_statistics(test_observations)
ood_results = runner._mahalanobis_reference.evaluate_batch(
    all_summaries, alpha=runner.config.mahalanobis_alpha
)

mahalanobis_stats = np.array([r.statistic for r in ood_results], dtype=float)
ood_rejected = np.array([r.rejected for r in ood_results], dtype=bool)
cutoff = float(ood_results[0].cutoff) if len(ood_results) > 0 else np.nan

mahalanobis_stats.shape, ood_rejected.sum(), cutoff

In [ ]:
train_stats = runner._mahalanobis_reference.train_statistics
all_summaries = runner._summary_statistics(test_observations)
ood_results = runner._mahalanobis_reference.evaluate_batch(
    all_summaries, alpha=runner.config.mahalanobis_alpha
)
test_stats = np.array([r.statistic for r in ood_results], dtype=float)
cutoff = runner._mahalanobis_reference.threshold(
    alpha=runner.config.mahalanobis_alpha
)

bins = np.linspace(
    min(train_stats.min(), test_stats.min()),
    max(train_stats.max(), test_stats.max()),
    40,
)

plt.figure(figsize=(8, 5))
plt.hist(
    train_stats, bins=bins, density=True, alpha=0.5, label="Train statistics"
)
plt.hist(
    test_stats, bins=bins, density=True, alpha=0.6, label="Test statistics"
)
plt.axvline(
    cutoff,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Cutoff = {cutoff:.3f}",
)
plt.xlabel("Mahalanobis statistic")
plt.ylabel("Density")
plt.title("Train vs Test Mahalanobis Statistics")
plt.legend()
plt.tight_layout()
plt.show()


## 5. PSIS diagnostics (manual)

Compute the target posterior log density for amortized samples, then compare to amortized log-proposal density with PSIS.


In [ ]:
log_post_vec = task.vectorized_log_posterior_fn(observation)
log_target = log_post_vec(q_samples)
psis = compute_psis(log_target=log_target, log_proposal=log_q)
psis.pareto_k, psis.ess

In [ ]:
posterior_draws_psis = resample_with_weights(
    samples=q_samples,
    weights=psis.smoothed_normalized_weights,
    num_draws=1000,
    replace=True,
)

In [ ]:
from amortized_bayesian_workflow.utils import corner_plot

_ = corner_plot(
    q_samples,
    posterior_draws_psis,
    labels=["amortized", "psis-corrected"],
    var_names=task.var_names,
    transform=task.transform_to_constrained_space,
)

## 6. Manual MCMC refinement (if needed)


In [ ]:
backend_name = "blackjax_chees_hmc"
# backend_name = "blackjax_nuts"
# backend_name = "tfp_chees_hmc"
iter_warmup = 200
if backend_name in ["blackjax_chees_hmc", "tf_chees_hmc"]:
    options = {"num_superchains": 16, "subchains_per_superchain": 128}
    iter_sampling = 1
    initial_positions = q_samples[: options["num_superchains"]]
elif backend_name == "blackjax_nuts":
    options = {"num_chains": 4}
    iter_sampling = 500
    initial_positions = q_samples[: options["num_chains"]]

log_post_single = task.single_log_posterior_fn(
    observation
)  # Use a non-vectorized log-prob function for MCMC sampling, since most MCMC backends expect that.
mcmc_result = run_mcmc(
    backend_name=backend_name,
    initial_positions=initial_positions,
    log_prob_fn=log_post_single,
    iter_warmup=iter_warmup,
    iter_sampling=iter_sampling,
    options=options,
    seed=seed,
)
mcmc_result.backend, mcmc_result.draws.shape, mcmc_result.diagnostics

print("MCMC draws shape:", mcmc_result.draws.shape)
print(
    f"{mcmc_result.rhat_name}: {mcmc_result.diagnostics.get(mcmc_result.rhat_name)}"
)

In [ ]:
posterior_draws_mcmc = mcmc_result.draws.reshape(
    -1, mcmc_result.draws.shape[-1]
)
posterior_draws_mcmc.shape

## 7. Compare with end-to-end workflow.run(...)

Once manual debugging looks good, switch back to the simpler high-level API.


In [ ]:
from pathlib import Path
from amortized_bayesian_workflow import (
    ArtifactLayout,
    InferenceConfig,
    InferenceRunner,
)
from amortized_bayesian_workflow.tasks.examples import PsychometricTask
from amortized_bayesian_workflow.utils import read_from_file
from amortized_bayesian_workflow.approximators import (
    BayesFlowAmortizedPosterior,
)

task = PsychometricTask()
layout = ArtifactLayout(
    root=Path("./artifacts"),
    task_name=task.task_name,
)
config = InferenceConfig(
    mcmc_backend="blackjax_nuts",
    seed=2026,
    rewrite_persisted_dataset_results=True,
    force_mcmc=True,
)

approximator = BayesFlowAmortizedPosterior.load(layout.models_dir)
runner = InferenceRunner(task=task, approximator=approximator, config=config)


test_observations = read_from_file(task.task_info_dir / "test_dataset.pkl")[
    :10
]
report = runner.run(test_observations[-2:])
report.summary_table()